# Greedy based agents

> Agents utelizing the Greedy approach for Dynamic pricing and learning problems

In [ ]:
#| default_exp agents.dynamic_pricing.greedy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction

In [ ]:
#| export
class GreedyPolicy():
    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = None,
                 actionprocessors: Optional[List[object]] = None,
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        assert type(alpha) == type(beta), "alpha and beta must be of the same type"
        if type(alpha) == None:
            alpha = np.zeros(environment_info.observation_space['features'].shape[0])   
            beta = np.zeros(environment_info.observation_space['features'].shape[0])
        if isinstance(ex_prices, list):
            ex_prices = np.array(ex_prices)
        assert ex_prices.shape[0] >= 2
        self.environment_info = environment_info
        self.ex_prices = ex_prices
        self.alpha = alpha
        self.beta = beta
        self.actionprocessors = actionprocessors
        self.price_function = price_function # Needs to return an np array
        self.g = g
        self.t = 0
        self.X = np.empty((0, environment_info.observation_space['features'].shape[0] * 2)) 
        self.Y = np.empty((0, 1))
        self.mode = "train"
        self.actionprocessors.append(ClipAction(environment_info.action_space.low, environment_info.action_space.high))

    def draw_action(self, observation: np.ndarray):
        if self.t < self.ex_prices.shape[0]:
            price = self.ex_prices[self.t]
        else:
            price = np.empty(0)
            X = observation['features']
            price = self.price_function(X, self.alpha, self.beta)
            
        for processor in self.actionprocessors:
            price = processor(price)
        
        return np.array(price)
    
    def fit(self, X, Y, action):
        assert self.mode == "train"
        self.t += 1
        X = np.concatenate([X, X * action])
        self.X = np.vstack([self.X, X])
        self.Y = np.vstack([self.Y, Y])
        self.parameter_update()
    
    def parameter_update(self):
        model = sm.OLS(self.Y, self.X)
        results = model.fit()
        self.alpha = results.params[:self.environment_info.observation_space['features'].shape[0]]
        self.beta = results.params[self.environment_info.observation_space['features'].shape[0]:]
    
    def update_task(self, env):
        self.t = 0
        self.environment_info = env.mdp_info
        self.X = np.empty((0, self.environment_info.observation_space['features'].shape[0] * 2))
        self.Y = np.empty((0, 1))
        self.actionprocessors[-1] = ClipAction(self.environment_info.action_space.low, self.environment_info.action_space.high)
        
    def reset(self):
        return

In [ ]:
#| export
class GreedyCoreAgent(Agent):

    """
    Base class for greedy bandit agents.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = [],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        
        policy = GreedyPolicy(environment_info=environment_info, obsprocessors=obsprocessors, actionprocessors=actionprocessors, ex_prices=ex_prices, alpha=alpha, beta=beta, price_function=price_function, g=g)
        self.agent_name = agent_name
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]["features"]
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
    
    def update_task(self, env):
        self.policy.update_task(env)
        

In [ ]:
#| export
class GreedyAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for GreedyCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = GreedyCoreAgent(environment_info = environment_info,
                                     obsprocessors = obsprocessors, 
                                     actionprocessors = actionprocessors, 
                                     agent_name = agent_name, 
                                     ex_prices = ex_prices, 
                                     alpha = alpha, 
                                     beta = beta, 
                                     price_function = price_function, 
                                     g = g)
        super().__init__(environment_info = environment_info, obsprocessors = obsprocessors, agent_name = agent_name)
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)
        